# Lesson 04 - Tool Use Design Pattern

यस पाठमा तपाईं Microsoft Agent Framework (Python) प्रयोग गरेर AI एजेन्टहरूको लागि **Tool Use** डिजाइन ढाँचा सिक्नुहुनेछ। हामीले समेटेका छौं:

- `@tool` डेकोरेटर र टाइप गरिएका प्यारामिटरहरूसहितको फंक्शन टुलहरू परिभाषित गर्नु
- मोडललाई प्रत्येक टुलले के गर्छ भन्ने थाहा पाउन टुल स्किमाहरू प्रदान गर्नु
- `approval_mode` संग टुल कार्यान्वयन नियन्त्रण गर्नु
- Pydantic मोडलहरू र `response_format` मार्फत **संरचित आउटपुट** फर्काउनु

परिदृश्य एक **यात्रा बुकिङ एजेन्ट** हो जसले गन्तव्यहरू खोज्न, उपलब्धता जाँच्न, र उडान जानकारी प्राप्त गर्न सक्छ।


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटरसँग उपकरणहरू परिभाषित गर्दै

`@tool` डेकोरेटरले एउटा सरल पाइथन कार्यलाई उपकरणमा परिणत गर्छ जुन एजेन्टले कल गर्न सक्छ।  
मुख्य बुँदाहरू:

- **डोकस्ट्रिंग** उपकरणको वर्णन हुन्छ जुन मोडलले देख्छ।  
- **टाइप एनोटेसनहरू** (वर्णनहरू सहित `Annotated` समावेश) उपकरणको स्किमालाई परिभाषित गर्छ।  
- `approval_mode` ले नियन्त्रण गर्छ कि प्रयोगकर्ताले प्रत्येक कल अघि स्वीकृति दिनैपर्छ वा होइन।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## एक जना एजेन्ट धेरै उपकरणहरूसित सिर्जना गर्दै

मोडेलले प्रयोगकर्ताको प्रश्नको जवाफ दिन आवश्यक पर्ने जुनसुकै उपकरणलाई बोलाउन सकून् भनेर तीनवटै उपकरणहरूलाई क्लाइन्टमा पठाउनुहोस्।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## उपकरणहरूसँग संरचित आउटपुट

`response_format` लाई Pydantic मोडेलमा सेट गरेर, एजेन्टलाई स्वतन्त्र रूपमा पाठको सट्टा राम्रो-टाइप गरिएको JSON वस्तु फर्काउन बाध्य पारिन्छ। जब डाउनस्ट्रीम कोडले परिणामलाई प्रोग्राम्याटिक रूपमा उपभोग गर्न आवश्यक हुन्छ तब यो उपयोगी हुन्छ।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## उपकरण स्वीकृति ढाँचा

`@tool` मा रहेको `approval_mode` प्यारामिटरले उपकरण कलहरू कार्यान्वयन हुनु अघि मानवीय स्वीकृतिको आवश्यक छ कि छैन भन्ने नियन्त्रण गर्छ:

| मोड | व्यवहार |
|---|---|
| `"never_require"` | उपकरण स्वचालित रूपमा चल्छ — प्रयोगकर्ताको पुष्टि आवश्यक छैन। |
| `"always_require"` | हरेक कल कार्यान्वयन हुनु अघि प्रयोगकर्ताबाट स्वीकृति हुनै पर्छ। |

तपाईंले ती उपकरणहरूका लागि `"always_require"` प्रयोग गर्नुहोस् जसले साइड-इफेक्ट्स गर्दछन् (जस्तै, फ्लाइट बुकिङ, क्रेडिट कार्ड चार्जिङ) ताकि मानव प्रक्रियामा रहोस।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## सारांश

यस पाठमा तपाईले सिक्नुभयो कसरी:

1. **उपकरणहरू परिभाषित गर्ने** `@tool` डेकोरेटर प्रयोग गरेर, टाइप गरिएको प्यारामिटरहरू र डकस्ट्रिङसँग जुन उपकरण स्कीमा को रूपमा सेवा गर्दछ।
2. **धेरै उपकरणहरू संयोजन गर्ने** ताकि एजेन्टले जटिल प्रश्नहरूको उत्तर दिन तिनीहरूलाई अनुक्रममा कल गर्न सक्छ।
3. **संरचित आउटपुट फर्काउने** पायडान्टिक मोडेललाई `response_format` को रूपमा पास गरेर।
4. **उपकरण स्वीकृति नियन्त्रण गर्ने** `approval_mode` सँग, संवेदनशील अपरेसनहरूको लागि मानवलाई प्रक्रिया भित्र राख्न।

यी ढाँचाहरू भरपर्दो, उत्पादन-तयार एजेन्टहरू विकास गर्ने आधार तयार पार्दछन् जसले बाह्य प्रणालीहरूसँग सुरक्षित रूपमा अन्तरक्रिया गर्न सक्छन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
यस कागजातलाई AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरी अनुवाद गरिएको हो। हामी सटीकता लागि प्रयासरत भए तापनि, कृपया बुझ्नुहोस् कि स्वचालित अनुवादहरूमा त्रुटि वा अकुरेसी हुनसक्छ। मूल कागजात त्यसको मौलिक भाषामा आधिकारिक स्रोत मानिनेछ। महत्वपूर्ण जानकारीका लागि, पेशेवर मानवीय अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न हुने कुनै पनि गलतफहमी वा भ्रमको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
